In [0]:
%sql
-- Builds the SALES_FACT gold table per spec Section 3:
-- Latest qualifying sale per LEAD_ID by ACTIVITY_AT
-- Qualifying activity types: 7) New Sale and 8) New Sale [Custom Payment Plan]
-- Includes: setter/closer identity (enriched from CRM users), contract value, date of sale, program
-- Excludes: cash collected
-- Dedup: ACTIVITY_ID latest by DATE_UPDATED DESC → INSERT_DATE DESC, then latest sale per LEAD_ID by ACTIVITY_AT
-- UNHARDCODED (Nov 9 2026): qualifying type ids resolve by NAME from the 2_4_01
-- dictionary (silver.custom_activites_processed); setter/closer/contract value/
-- program cf_ keys resolve by field NAME from the 2_4_02 fields child
-- (silver.custom_activites_processed_fields). Close regenerating any id no
-- longer breaks this build.
CREATE OR REPLACE TABLE crm_ingestion.gold.sales_fact AS (
    WITH qualifying_types AS (
        SELECT id AS type_id
        FROM crm_ingestion.silver.custom_activites_processed
        WHERE name IN ('7) New Sale', '8) New Sale [Custom Payment Plan]')
    ),
    field_defs AS (
        SELECT custom_activites_processed_id AS type_id,
               id AS field_id,
               name AS field_name
        FROM crm_ingestion.silver.custom_activites_processed_fields
        WHERE name IN ('Setter', 'Closer', 'Contract Value', 'Program')
    ),
    deduped_sales AS (
        SELECT
            id AS activity_id,
            lead_id,
            activity_at AS date_of_sale,
            custom_activity_type_id,
            created_by,
            custom_fields,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS dedup_rn
        FROM crm_ingestion.silver.lead_activites_processed
        JOIN qualifying_types q ON custom_activity_type_id = q.type_id
    ),
    latest_sale AS (
        SELECT
            activity_id,
            lead_id,
            date_of_sale,
            custom_activity_type_id,
            created_by,
            custom_fields,
            ROW_NUMBER() OVER(PARTITION BY lead_id ORDER BY date_of_sale DESC) AS sale_rn
        FROM deduped_sales
        WHERE dedup_rn = 1
    )
    SELECT
        s.lead_id,
        s.activity_id,
        s.date_of_sale,
        s.custom_activity_type_id AS sale_type_id,
        element_at(s.custom_fields, f_setter.field_id) AS setter_id,
        COALESCE(setter_u.csm_name, element_at(s.custom_fields, f_setter.field_id)) AS setter_name,
        element_at(s.custom_fields, f_closer.field_id) AS closer_id,
        COALESCE(closer_u.csm_name, element_at(s.custom_fields, f_closer.field_id)) AS closer_name,
        element_at(s.custom_fields, f_contract.field_id) AS contract_value,
        element_at(s.custom_fields, f_program.field_id) AS program,
        s.created_by,
        COALESCE(created_u.csm_name, s.created_by) AS created_by_name
    FROM latest_sale s
    LEFT JOIN field_defs f_setter
        ON f_setter.type_id = s.custom_activity_type_id AND f_setter.field_name = 'Setter'
    LEFT JOIN field_defs f_closer
        ON f_closer.type_id = s.custom_activity_type_id AND f_closer.field_name = 'Closer'
    LEFT JOIN field_defs f_contract
        ON f_contract.type_id = s.custom_activity_type_id AND f_contract.field_name = 'Contract Value'
    LEFT JOIN field_defs f_program
        ON f_program.type_id = s.custom_activity_type_id AND f_program.field_name = 'Program'
    LEFT JOIN crm_ingestion.gold.staff_dim setter_u
        ON element_at(s.custom_fields, f_setter.field_id) = setter_u.csm_id
    LEFT JOIN crm_ingestion.gold.staff_dim closer_u
        ON element_at(s.custom_fields, f_closer.field_id) = closer_u.csm_id
    LEFT JOIN crm_ingestion.gold.staff_dim created_u
        ON s.created_by = created_u.csm_id
    WHERE s.sale_rn = 1
)

In [0]:
%sql
SELECT count(*)
FROM crm_ingestion.gold.sales_fact

In [0]:
%sql
-- VALIDATION — qualifying-sale detection. Spec: latest sale per lead from the two
-- qualifying outcomes ("7) New Sale", "8) New Sale [Custom Payment Plan]").
-- Row set 1: every sale/payment-named custom activity type with its activity volume
-- (catches qualifying sales recorded under a DIFFERENT type id).
-- Row set 2: current clients with contract value on file but NO qualifying sale row.
WITH type_meta AS (
    SELECT DISTINCT id, name FROM crm_ingestion.silver.custom_activites_processed
),
type_counts AS (
    SELECT custom_activity_type_id, COUNT(DISTINCT id) AS activities, COUNT(DISTINCT lead_id) AS leads
    FROM crm_ingestion.silver.lead_activites_processed
    WHERE custom_activity_type_id IS NOT NULL
    GROUP BY 1
),
gap AS (
    SELECT d.lead_id
    FROM crm_ingestion.gold.lead_dim d
    LEFT JOIN crm_ingestion.gold.sales_fact s ON s.lead_id = d.lead_id
    WHERE s.lead_id IS NULL
      AND TRY_CAST(REGEXP_REPLACE(d.Contracted_Value, '[^0-9.]', '') AS DOUBLE) > 0
),
backend_sale_leads AS (
    SELECT DISTINCT a.lead_id
    FROM crm_ingestion.silver.lead_activites_processed a
    JOIN type_meta m ON m.id = a.custom_activity_type_id
    WHERE m.name = '14) Backend Sale'
)
SELECT
    COALESCE(m.name, tc.custom_activity_type_id)              AS activity_type,
    tc.activities,
    tc.leads
FROM type_counts tc
LEFT JOIN type_meta m ON m.id = tc.custom_activity_type_id
WHERE m.name ILIKE '%sale%' OR m.name ILIKE '%payment%'
   OR tc.custom_activity_type_id IN (
       SELECT id FROM crm_ingestion.silver.custom_activites_processed
       WHERE name IN ('7) New Sale', '8) New Sale [Custom Payment Plan]')
   )
UNION ALL
SELECT 'GAP: contract value on file, NO qualifying sale row' AS activity_type,
       COUNT(*) AS activities, NULL AS leads
FROM gap
UNION ALL
SELECT 'GAP subset: sale recorded as 14) Backend Sale on own lead', COUNT(DISTINCT g.lead_id), NULL
FROM gap g
JOIN backend_sale_leads b ON b.lead_id = g.lead_id
UNION ALL
SELECT 'GAP subset: NO custom activity of ANY type on own lead', COUNT(*), NULL
FROM gap g
WHERE NOT EXISTS (
    SELECT 1 FROM crm_ingestion.silver.lead_activites_processed a
    WHERE a.lead_id = g.lead_id AND a.custom_activity_type_id IS NOT NULL
)
UNION ALL
SELECT 'GAP subset: qualifying sale on a SIBLING lead (same person, via spine)', COUNT(DISTINCT g.lead_id), NULL
FROM gap g
JOIN crm_ingestion.silver.user_identity_map sp ON ARRAY_CONTAINS(sp.lead_ids, g.lead_id)
JOIN crm_ingestion.gold.sales_fact sf        ON ARRAY_CONTAINS(sp.lead_ids, sf.lead_id)
WHERE sf.lead_id != g.lead_id

In [0]:
%sql
SELECT *
FROM crm_ingestion.gold.sales_fact
ORDER BY date_of_sale DESC
LIMIT 100